## The cluster PKI — who has what certificate

A cluster is held together by TLS — almost every component talks to every other over HTTPS, and identities are proven with X.509 client certificates.

**Two CAs by default** (a kubeadm cluster):

- **Kubernetes CA** (`/etc/kubernetes/pki/ca.crt`) — signs the front-end certs (API server, kubelet clients).
- **etcd CA** (`/etc/kubernetes/pki/etcd/ca.crt`) — signs etcd's server + peer certs.

(A third `front-proxy-ca` serves the aggregation layer — metrics-server, extension APIs.)

Certs by role, in `/etc/kubernetes/pki/`: `apiserver.crt` (API server TLS), `apiserver-kubelet-client.crt` (API server → kubelet), `apiserver-etcd-client.crt` (API server → etcd), `etcd/server.crt` + `etcd/peer.crt`. Each node's kubelet has its own cert (`/var/lib/kubelet/pki/...`), auto-rotated before expiry.

Components read **kubeconfig** files that bundle cert + key + API server URL: `admin.conf` (for humans — never commit), `kubelet.conf`, `controller-manager.conf`, `scheduler.conf`. Your `~/.kube/config` is the same format.

**Certs expire after 1 year** by default. Renew before then:

```bash
kubeadm certs check-expiration     # what expires when
kubeadm certs renew all            # renew all; needs a control-plane restart
```

**The CKA loves this:** "the cluster stopped working a year after install — why?" Answer: **certs expired.** (`kubeadm upgrade` rotates them as a side effect.) On our map this section frames the **kube-apiserver** — the TLS hub every certificate ultimately serves.